# Credit Card Fraud: Exploratory Data Analysis

**Course:** MAT 311, Data Science with AI assignment

**AI system used:** Claude Sonnet 5.5

**Dataset:** `card_transdata.csv`, 999 credit card transactions. The target column `fraud` is 1 if the transaction was fraudulent and 0 otherwise.

Each prompt I used appears in a markdown cell above the code it produced.

I have a CSV named card_transdata.csv of credit card transactions with a 0/1 column called fraud. Write pandas code to load it, then show me the shape, the first few rows, and the data type of each column.

AI: Claude Sonnet 5.5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("card_transdata.csv")
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.dtypes

This output shows that the dataset was successfully loaded into Python using pandas. The Shape command displays the total number of rows and columns, while df.head() shows the first five rows of the dataset to give an overview of the data.

Count the missing values in each column of the dataframe, show it as a count and a percentage, and tell me how many rows have at least one missing value.

AI: Claude Sonnet 5.5

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
print(missing)
print("\nTotal missing cells:", int(df.isna().sum().sum()))
print("Rows with at least one missing value:", int(df.isna().any(axis=1).sum()), "of", len(df))

This output shows how many missing values are in each column of the dataset and what percentage of each column is missing. The total missing cells and the number of rows with missing values help determine whether the dataset is complete or needs cleaning before analysis. If all the counts are zero, it means there are no missing values in the dataset.

 Make a cleaned copy of the dataframe that drops rows with missing values. Tell me how many rows were removed, and check that the fraud rate did not change much after dropping them.

AI: Claude Sonnet 5.5

In [ ]:
df_clean = df.dropna().copy()

print("Rows before:", len(df))
print("Rows after: ", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))
print()
print("Fraud rate before dropping: {:.2f}%".format(df["fraud"].mean() * 100))
print("Fraud rate after dropping:  {:.2f}%".format(df_clean["fraud"].mean() * 100))

This output shows how many rows were removed from the dataset after dropping missing values. It also compares the fraud rate before and after cleaning to see whether removing incomplete rows changed the percentage of fraudulent transactions. If the fraud rates are close, it suggests that dropping missing values did not significantly affect the fraud data.

Show summary statistics for every column. Also show the skewness of the three continuous columns, since some of them look like they have extreme outliers.

AI: Claude Sonnet 5.5

In [ ]:
print(df_clean.describe().T.round(3))
print()
print("Skewness of the continuous columns:")
cont_cols = ["distance_from_home", "distance_from_last_transaction", "ratio_to_median_purchase_price"]
print(df_clean[cont_cols].skew().round(2))

This output shows the descriptive statistics for the dataset, including the average, minimum, maximum, and standard deviation for each column. The skewness values show whether the continuous variables are evenly distributed or have values clustered more toward one side. A positive skew means there are some unusually high values compared to most of the data.

 Show how many transactions are fraud vs not fraud, with percentages, and make a bar chart of it with the counts labeled on the bars.

AI: Claude Sonnet 5.5

In [ ]:
counts = df_clean["fraud"].value_counts().sort_index()
percents = (df_clean["fraud"].value_counts(normalize=True).sort_index() * 100).round(2)
print(pd.DataFrame({"count": counts, "percent": percents}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df_clean, x="fraud", ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Not fraud (0)", "Fraud (1)"])
ax.set_title("Class balance")
ax.set_ylabel("Number of transactions")
for container in ax.containers:
    ax.bar_label(container)
plt.show()

This output shows the number and percentage of fraudulent and non-fraudulent transactions in the dataset. The countplot makes it easier to compare the two groups visually and see whether the data is balanced or imbalanced. If there are many more non-fraudulent transactions than fraudulent ones, it means fraud is less common in the dataset.

Make histograms of the three continuous columns. Because they are heavily right-skewed, use a log scale on the x-axis so the shape is visible.

AI: Claude Sonnet 5.5

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cont_cols):
    positive = df_clean[col][df_clean[col] > 0]
    sns.histplot(positive, bins=40, log_scale=True, ax=ax)
    ax.set_title(col)
    ax.set_xlabel(col + " (log scale)")
plt.tight_layout()
plt.show()

This output shows the distribution of the three continuous variables: distance from home, distance from the last transaction, and ratio to median purchase price. The histograms use a logarithmic scale to make it easier to see the spread of the data and identify unusually high values. The graphs help show whether most transactions have smaller values while a few have much larger values.

Compare the three continuous columns between fraud and non-fraud transactions. Give me the mean and median for each group, and boxplots on a log y-axis.

AI: Claude Sonnet 5.5

In [ ]:
print(df_clean.groupby("fraud")[cont_cols].agg(["mean", "median"]).round(3).T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cont_cols):
    positive = df_clean[df_clean[col] > 0]
    sns.boxplot(data=positive, x="fraud", y=col, ax=ax)
    ax.set_yscale("log")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Not fraud", "Fraud"])
    ax.set_title(col + " by fraud status")
plt.tight_layout()
plt.show()

This output compares the mean and median values of the three continuous variables for fraudulent and non-fraudulent transactions. The boxplots show how the values differ between the two groups and help identify patterns or unusually large values. If fraudulent transactions have higher distances or purchase price ratios, these variables may be useful for identifying potential fraud.

For each of the four 0/1 columns (repeat_retailer, used_chip, used_pin_number, online_order), calculate the fraud rate for each value and show it as a bar chart.

AI: Claude Sonnet 5.5

In [ ]:
binary_cols = ["repeat_retailer", "used_chip", "used_pin_number", "online_order"]

rates = {col: (df_clean.groupby(col)["fraud"].mean() * 100).round(2) for col in binary_cols}
print(pd.DataFrame(rates).T.rename(columns={0.0: "fraud rate % when 0", 1.0: "fraud rate % when 1"}))

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, col in zip(axes, binary_cols):
    sns.barplot(x=rates[col].index.astype(int), y=rates[col].values, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("value")
axes[0].set_ylabel("Fraud rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Group sizes matter: a rate based on very few rows is unreliable.
for col in binary_cols:
    print(col)
    print(df_clean.groupby(col)["fraud"].agg(rows="count", frauds="sum"))
    print()

This output compares fraud rates for different transaction behaviors, including repeat retailers, chip usage, PIN usage, and online orders. The bar charts show which behaviors are associated with higher fraud rates, while the group sizes show how many transactions fall into each category. This helps identify which factors may be useful for predicting fraud, while keeping in mind that smaller groups may have less reliable results.

Make a correlation heatmap of all the columns with the correlation values printed in each cell, and tell me which columns correlate most strongly with fraud.

AI: Claude Sonnet 5.5

In [ ]:
corr = df_clean.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

print("Correlation with fraud, strongest first:")
print(corr["fraud"].drop("fraud").sort_values(key=abs, ascending=False).round(3))

This output shows the correlation between all the variables in the dataset and how strongly they are related to each other. The heatmap makes it easier to see positive and negative relationships, while the printed list ranks which variables have the strongest correlation with fraud. Variables with stronger correlations may be useful for identifying fraudulent transactions, but correlation does not necessarily mean causation.